In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, SimpleRNN
from tensorflow.keras.metrics import SparseTopKCategoricalAccuracy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.activations import leaky_relu
import keras_tuner as kt

In [79]:
STEP = 7

In [80]:
X = np.loadtxt(f'../data/features/step{STEP}_features.csv', delimiter=',')
y = np.loadtxt(f'../data/features/step{STEP}_label.csv', delimiter=',')

print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (211, 781)
y shape: (211,)


In [81]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=5)

print('X Train shape:', X_train.shape)
print('X Test shape:', X_test.shape)
print('y Train shape:', y_train.shape)
print('y Test shape:', y_test.shape)

X Train shape: (189, 781)
X Test shape: (22, 781)
y Train shape: (189,)
y Test shape: (22,)


In [71]:
ann = Sequential([
  Input(shape=(X_train.shape[1],)),
  Dense(512, activation='relu'),
  Dense(256, activation='relu'),
  Dense(128, activation='softmax')
])
ann.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 512)            │       400,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 564,608 (2.15 MB)

 Trainable params: 564,608 (2.15 MB)

 Non-trainable params: 0 (0.00 B)

In [72]:
ann.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(), metrics=['accuracy', SparseTopKCategoricalAccuracy(k=5)])
ann.fit(X_train, y_train, epochs=100, validation_split=0.2)

Epoch 1/100


5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 101ms/step - accuracy: 0.0235 - loss: 4.7841 - sparse_top_k_categorical_accuracy: 0.0783 - val_accuracy: 0.0263 - val_loss: 4.6771 - val_sparse_top_k_categorical_accuracy: 0.0789
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.1017 - loss: 4.1088 - sparse_top_k_categorical_accuracy: 0.3773 - val_accuracy: 0.0263 - val_loss: 4.7836 - val_sparse_top_k_categorical_accuracy: 0.0789
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.1108 - loss: 3.7008 - sparse_top_k_categorical_accuracy: 0.3592 - val_accuracy: 0.0263 - val_loss: 5.0022 - val_sparse_top_k_categorical_accuracy: 0.0789
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.2062 - loss: 3.3682 - sparse_top_k_categorical_accuracy: 0.4524 - val_accuracy: 0.0263 - val_loss: 5.1176 - val_sparse_top_k_categorical_accuracy: 0.0789
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3653 - loss: 2.9857 - sparse_top_k_categorical_accuracy: 0.6643 - val_accur

In [73]:
ann.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.1364 - loss: 7.3509 - sparse_top_k_categorical_accuracy: 0.1818


[7.35092306137085, 0.13636364042758942, 0.1818181872367859]

### Tuning Hyperparameters

In [74]:
def build_model(hp):
  model = Sequential()
  model.add(Input(shape=(781,)))

  # Hidden Layer 1
  model.add(Dense(
    units=hp.Choice('units1', [128, 256, 512, 1024, 2048]),
    activation='relu',
  ))

  # Dropout Layer 1 (Optional)
  if hp.Boolean('use_dropout1_layer'):
    model.add(Dropout(hp.Float('dropout1', min_value=0.1, max_value=0.5, step=0.1)))

  # Hidden Layer 2 (Optional)
  if hp.Boolean('use_hidden_layer2'):
    model.add(Dense(
      units=hp.Choice('units2', [128, 256, 512, 1024, 2048]),
      activation='relu'
    ))

  # Dropout Layer 2 (Optional)
  if hp.Boolean('use_dropout2_layer'):
    model.add(Dropout(hp.Float('dropout2', min_value=0.1, max_value=0.5, step=0.1)))

  # Output Layer
  model.add(Dense(128, activation='softmax'))

  # Optimizer and Learning Rate
  lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
  optimizer = Adam(learning_rate=lr)

  model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', SparseTopKCategoricalAccuracy(k=5)]
  )

  return model

In [82]:
tuner = kt.RandomSearch(
  build_model,
  objective='val_accuracy',
  max_trials=20,
  executions_per_trial=1,
  directory='../tuning_dir',
  project_name=f'_step{STEP}'
)

tuner.search(X_train, y_train, epochs=100, validation_split=0.1)

Reloading Tuner from ../tuning_dir\_step7\tuner0.json


In [83]:
best_model = tuner.get_best_models()[0]
best_hps = tuner.get_best_hyperparameters()[0]

print("Best Hyperparameters:")
print(f"Units1: {best_hps['units1']}")
print(f"Use Dropout1 Layer: {best_hps['use_dropout1_layer']}")
if best_hps['use_dropout1_layer']:
    print(f"Dropout1: {best_hps['dropout1']}")
print(f"Use Second Layer: {best_hps['use_hidden_layer2']}")
if best_hps['use_hidden_layer2']:
    print(f"Units2: {best_hps['units2']}")
print(f"Use Dropout2 Layer: {best_hps['use_dropout2_layer']}")
if best_hps['use_dropout2_layer']:
    print(f"Dropout2: {best_hps['dropout2']}")
print(f"Learning Rate: {best_hps['learning_rate']}")

Best Hyperparameters:
Units1: 256
Use Dropout1 Layer: True
Dropout1: 0.4
Use Second Layer: True
Units2: 256
Use Dropout2 Layer: False
Learning Rate: 0.01


In [84]:
best_model.evaluate(X_test, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 786ms/step - accuracy: 0.2727 - loss: 3.5636 - sparse_top_k_categorical_accuracy: 0.8636


[3.563605546951294, 0.27272728085517883, 0.8636363744735718]

In [85]:
best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]

print("Best trial ID:", best_trial.trial_id)
print("Best val_accuracy:", best_trial.score)
print("Best hyperparameters:", best_trial.hyperparameters.values)


Best trial ID: 11
Best val_accuracy: 0.5263158082962036
Best hyperparameters: {'units1': 256, 'use_dropout1_layer': True, 'use_hidden_layer2': True, 'use_dropout2_layer': False, 'learning_rate': 0.01, 'units2': 256, 'dropout2': 0.1, 'dropout1': 0.4}
